# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

It also showcases the **BioCypher Agent Graph API** — a pure-Python
graph with built-in type-aware queries, path finding, and JSON
serialisation that can be injected directly into LLM prompts.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- Ollama running locally with `deepseek-r1:14b` pulled

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [12]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 212 nodes, 639 edges

Micro-CKG: 212 nodes, 639 edges


## 5.2 Create QA Agent (Ollama — deepseek-r1:14b)

The QA agent uses **deepseek-r1:14b** running locally via Ollama with a pruned graph context. Weak edges (|log2FC| < 0.25 and spatial correlation < 0.4) are removed to reduce prompt token size. No API key needed — runs entirely on-device.

In [13]:
import time

# Aggressively prune the graph to reduce LLM prompt token size,
# but always preserve drug translational edges (they have no log2fc/spatial attrs)
_KEEP_EDGE_LABELS = {"drug_gene_association", "gene_associated_with_disease"}

optimized_graph = graph.copy()
edges_to_remove = [
    (u, v) for u, v, d in optimized_graph.edges(data=True)
    if d.get("label") not in _KEEP_EDGE_LABELS
    and abs(d.get("log2fc", 0)) < 0.25
    and d.get("spatial_correlation", 0) < 0.4
]
optimized_graph.remove_edges_from(edges_to_remove)

print(f"Original edges: {graph.number_of_edges()}")
print(f"Pruned edges for LLM: {optimized_graph.number_of_edges()}")

try:
    agent = create_qa_agent(optimized_graph, provider="ollama", model="deepseek-r1:14b")
    print("QA agent initialised (Ollama — deepseek-r1:14b).")
    _OLLAMA_AVAILABLE = True
except Exception as _ollama_err:
    print(f"⚠ Ollama not available: {_ollama_err}")
    print("  LLM query cells below will be skipped.")
    _OLLAMA_AVAILABLE = False
    agent = None

Original edges: 639
Pruned edges for LLM: 489
QA agent initialised (Ollama — deepseek-r1:14b).


## 5.3 AD-Relevant Translational Queries

Four questions that probe the disease mechanisms surfaced by
Stabl feature selection (Notebook 02) and encoded in the Micro-CKG
(Notebook 04).  Each targets a different data-supported AD theme:

1. **Aβ–PrP axis** — drugs and diseases linked to the amyloid-β receptor Prnp
2. **Monoaminergic neurodegeneration** — Th as the highest-degree hub gene; drug landscape
3. **Direct AD-associated signalling** — Cdc42ep1’s Alzheimer’s disease link and neighbouring pathways
4. **Cross-theme integration** — which single gene bridges the most diseases, drugs, and regions?

In [14]:
if _OLLAMA_AVAILABLE:
    print("[Query 1] A\u03b2\u2013PrP axis: drugs targeting amyloid-\u03b2 synaptic toxicity...")
    print(query_graph(
        agent,
        "Prnp (PrP^C) is the primary neuronal receptor for amyloid-\u03b2 oligomers in the "
        "PSAPP Alzheimer's model. Using only edges present in the Micro-CKG, trace the "
        "complete evidence path from the mouse gene Prnp to its human ortholog PRNP, then "
        "to any connected drug nodes (ChEMBL). For each drug, report the mechanism of action, "
        "clinical phase, and the disease associations linked to PRNP. Cite every edge.",
    ))
else:
    print("Skipped — Ollama not available.")

[Query 1] Aβ–PrP axis: drugs targeting amyloid-β synaptic toxicity...
  Querying agent: Prnp (PrP^C) is the primary neuronal receptor for amyloid-β oligomers in the PSA...
Based on the provided Micro-CKG data, there is no direct or indirect evidence connecting the mouse gene Prnp to its human ortholog PRNP. The dataset does not include entries for Prnp or PRNP, nor are there any edges that would allow such a connection. Therefore, it is not possible to trace a complete evidence path from Prnp to PRNP using the provided data. Consequently, no connected drug nodes from ChEMBL can be identified in this context.

**Answer:**
There is no evidence in the provided Micro-CKG data connecting the mouse gene Prnp to its human ortholog PRNP. As a result, no further connections to drug nodes can be made based on the given dataset.


In [15]:
if _OLLAMA_AVAILABLE:
    print("\n[Query 2] Monoaminergic neurodegeneration (Th)...")
    print(query_graph(
        agent,
        "Th (Tyrosine Hydroxylase) is the highest-degree gene in the Micro-CKG with "
        "40 edges, including 15 drug associations. TH loss in the locus coeruleus is "
        "one of the earliest neuropathological events in Alzheimer's disease. Using "
        "only edges present in the graph: (a) list all drug nodes connected to gene:Th "
        "with their mechanism of action and clinical phase, (b) list all disease "
        "associations, and (c) identify which brain regions and cell types Th is "
        "differentially expressed in. Cite every edge with scores.",
    ))
else:
    print("Skipped — Ollama not available.")


[Query 2] Monoaminergic neurodegeneration (Th)...
  Querying agent: Th (Tyrosine Hydroxylase) is the highest-degree gene in the Micro-CKG with 40 ed...
**Answer:**

**Part (a): Drug Nodes Connected to Th**

- **Conclusion:** There are no drug nodes directly connected to Th in the provided graph.

**Part (b): Disease Associations for Th**

1. **Dystonia 5** (MONDO_0007495)
2. **Th tyrosine hydroxylase deficiency, Dopa-responsive dystonia** (MONDO_0011551)
3. **Dopa-responsive dystonia** (MONDO_0016812)
4. **Tyrosine hydroxylase deficiency** (MONDO_0100064)
5. **Autosomal recessive dopa-responsive dystonia** (Orphanet_101150)

**Part (c): Brain Regions and Cell Types with Differential Expression of Th**

- **Brain Regions:**
  - **White Matter** (spatial_correlation=0.7191)
  - **Cerebellum** (mean_expression=0.0766, log2fc=1.0616, pval_adj=0.000384)
  - **Hippocampus** (mean_expression=0.0036, log2fc=-3.5433, pval_adj=0.005756)
  - **Cortex** (mean_expression=0.0065, log2fc=-2.6903, pv

In [16]:
if _OLLAMA_AVAILABLE:
    print("\n[Query 3] Direct AD-associated gene (Cdc42ep1)...")
    print(query_graph(
        agent,
        "Cdc42ep1 carries a direct Alzheimer's disease association in OpenTargets, "
        "alongside Parkinson's disease, multiple sclerosis, and other neurodegenerative "
        "conditions. Using the Micro-CKG, identify: (a) all disease nodes connected to "
        "gene:Cdc42ep1 and their association scores, (b) which cell types and brain "
        "regions Cdc42ep1 is differentially expressed in, and (c) any pathway or drug "
        "nodes in its neighbourhood. Explain how this gene's graph context supports "
        "its role as a neurodegeneration hub. Cite each edge.",
    ))
else:
    print("Skipped — Ollama not available.")


[Query 3] Direct AD-associated gene (Cdc42ep1)...
  Querying agent: Cdc42ep1 carries a direct Alzheimer's disease association in OpenTargets, alongs...
The question regarding the gene Cdc42ep1 cannot be fully addressed using the provided dataset, as Cdc42ep1 is not included in the shared information. Here's a structured summary of the situation and potential next steps:

1. **Gene Absence in Dataset**: Cdc42ep1 is not present in the provided data, which includes other genes like SDC42EP1, SDC42EP2, etc. This absence means the specific details requested cannot be extracted from the given information.

2. **Next Steps**:
   - **Verify Gene Name**: Check if there was a typo or if the gene name should be different (e.g., Cdc42ep1 vs. another form).
   - **Use External Resources**: Consider using platforms like OpenTargets or Micro-CKG for detailed information on Cdc42ep1's associations, cell type expressions, and pathways.
   - **Literature Review**: Look into scientific literature to gat

In [17]:
if _OLLAMA_AVAILABLE:
    print("\n[Query 4] Cross-theme hub gene integration...")
    print(query_graph(
        agent,
        "Considering the three AD-relevant themes in this Micro-CKG (A\u03b2 synaptic "
        "toxicity via Prnp/Ngfr, monoaminergic neurodegeneration via Th, and direct "
        "AD-associated signalling via Cdc42ep1), which single gene node has the highest "
        "PageRank centrality and connects to the most disease, drug, and anatomical-region "
        "nodes? Explain why this gene is the strongest candidate for a multi-target "
        "therapeutic strategy in Alzheimer's disease. Cite all supporting edges.",
    ))
else:
    print("Skipped — Ollama not available.")


[Query 4] Cross-theme hub gene integration...
  Querying agent: Considering the three AD-relevant themes in this Micro-CKG (Aβ synaptic toxicity...
**Answer:**

The gene with the highest PageRank centrality in the Alzheimer's Disease (AD) Micro-CKG is **Syt2 (Syntaphilin)**. 

**Reasoning:**

1. **High Number of Connections:**
   - **Disease Connections:** Syt2 is linked to multiple diseases, including botulism and various myasthenic syndromes, indicating a broad role in disease mechanisms.
   - **Anatomical-Region Connections:** It connects to the White Matter region, which is significant in brain function and disease.
   - **Drug Connections:** While not explicitly listed, Syt2's involvement in synaptic toxicity suggests potential drug relevance.

2. **Strategic Importance:**
   - Syt2 is part of the Aβ synaptic toxicity pathway, a major theme in AD research. This pathway is critical as it involves synaptic damage, a key factor in AD progression.

3. **Central Role in Pathology:**
 

## 5.4 BioCypher Agent Graph — Structured Exploration

The **Agent Graph** (`biocypher.Graph`) provides type-aware indexing,
path finding, subgraph extraction, and lightweight JSON serialisation
without the overhead of a full DBMS. This section loads the Micro-CKG
via the Agent Graph API and demonstrates its built-in query methods.

In [18]:
from src.biocypher_adapter import load_graph as _unused, build_micro_ckg_agent
from src.spatial_pipeline import load_adata, run_stabl_cached, compute_clusters, annotate_clusters

# Rebuild Agent Graph from the same data used in notebook 04
_adata = load_adata(PROJECT_ROOT / "data" / "processed" / "ad_preprocessed.h5ad")
_stabl = run_stabl_cached(
    _adata, cache_dir=CACHE_DIR, dataset_name="geo_ad",
    label_method="condition", n_bootstraps=50, prefilter="de",
)
_adata = compute_clusters(_adata, n_hvgs=2000)
_cluster_ann = annotate_clusters(_adata)

# Fetch same enrichment data
from src.external_knowledge import map_orthologs, run_go_enrichment, get_disease_associations, get_string_ppi, get_drug_targets

_ortho_df = map_orthologs(_stabl["selected_genes"])
_human_genes = _ortho_df["human_symbol"].dropna().tolist()
_ortho_map = {
    str(r["mouse_symbol"]): str(r["human_symbol"])
    for _, r in _ortho_df.iterrows()
    if r.get("mouse_symbol") and r.get("human_symbol")
}
_ensembl_map = {
    str(r["human_symbol"]): str(r["ensembl_gene"])
    for _, r in _ortho_df.iterrows()
    if r.get("human_symbol") and r.get("ensembl_gene")
}
_enrich_df = run_go_enrichment(_human_genes)
_disease_df = get_disease_associations(_human_genes, ensembl_map=_ensembl_map)
_ppi_df = get_string_ppi(_human_genes)
_drug_df = get_drug_targets(_human_genes)

agent_graph = build_micro_ckg_agent(
    stabl_result=_stabl, adata=_adata,
    cluster_annotation=_cluster_ann, min_genes=20,
    ortho_map=_ortho_map, enrich_df=_enrich_df,
    disease_df=_disease_df, ppi_df=_ppi_df,
    drug_df=_drug_df,
)

  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/ad_preprocessed.h5ad
  Shape: 15687 spots × 22265 genes
  Loading cached Stabl results: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_results_3f25e12aeadc.pkl
  Selected 2000 highly variable genes (requested 2000)


/Users/shaunfchen/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
python(38140) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Leiden clustering: 18 clusters
  Cluster annotation: {'White_Matter': np.int64(11), 'Cortex': np.int64(3), 'Cerebellum': np.int64(2), 'Thalamus': np.int64(1), 'Hippocampus': np.int64(1)}
  Orthologs loaded from cache (31 mappings)
  GO enrichment loaded from cache
  Disease associations loaded from cache (145 entries)
  STRING PPI loaded from cache (7 interactions)
  Drug targets loaded from cache (32 entries)
  Running DE testing (Wilcoxon rank-sum)...
  DE results: 594 tests, 312 significant
  Building BioCypher Agent Graph...
  Agent Graph: 212 nodes ({'gene': 33, 'cell_type': 18, 'anatomical_entity': 5, 'drug': 32, 'biological_process': 3, 'disease': 121})
  Agent Graph: 639 edges ({'gene_cell_type_association': 312, 'gene_anatomical_entity_association': 118, 'cell_type_anatomical_entity_association': 18, 'drug_gene_association': 32, 'gene_participates_in_pathway': 9, 'gene_associated_with_disease': 143, 'gene_interacts_with_gene': 7})


### Type-Aware Queries & Path Finding

The Agent Graph lets you query by node type and find paths between
any two nodes — useful for building focused LLM prompts.

In [19]:
# Query by type
all_nodes = agent_graph.get_nodes()
gene_nodes = [n for n in all_nodes if n.type == "gene"]
pathway_nodes = [n for n in all_nodes if n.type == "biological_process"]
disease_nodes = [n for n in all_nodes if n.type == "disease"]
print(f"Agent Graph contents:")
print(f"  Genes: {len(gene_nodes)}")
print(f"  Pathways: {len(pathway_nodes)}")
print(f"  Diseases: {len(disease_nodes)}")

# Find paths
if gene_nodes and pathway_nodes:
    src = gene_nodes[0].id
    tgt = pathway_nodes[0].id
    paths = agent_graph.find_paths(src, tgt, max_length=3)
    print(f"\nPaths from {src} → {tgt}:")
    if paths:
        for edge in paths[0]:
            print(f"  {edge}")
    else:
        print("  No path found (max_length=3)")

# JSON context for LLM injection
json_ctx = agent_graph.to_json()
print(f"\nJSON context size: {len(json_ctx):,} chars (ready for LLM prompt injection)")

Agent Graph contents:
  Genes: 33
  Pathways: 3
  Diseases: 121

Paths from gene:Cyp27a1 → pathway:ppar_signaling_pathway:
  Edge(id='edge:gene_pathway:Cyp27a1_ppar_signaling_pathway', type='gene_participates_in_pathway', source='gene:Cyp27a1', target='pathway:ppar_signaling_pathway', properties={'pval_adj': 0.0103462499252384, 'combined_score': 255.3196691383704, 'source_library': 'KEGG_2021_Human'})

JSON context size: 239,595 chars (ready for LLM prompt injection)
